<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/01_importar_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/01_importar_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Ingestão de Dados do Kaggle para o BigQuery
**Dataset: Mercado de Games**

Este notebook é o primeiro passo da nossa arquitetura de BI. Ele automatiza o download, limpeza e envio dos dados para a camada RAW (Bronze) do BigQuery.

## 1. Importação das Bibliotecas

In [7]:
import pandas as pd
import os
import kagglehub
from google.cloud import bigquery
from google.colab import auth

# Autenticação no Google Colab
auth.authenticate_user()

## 2. Configuração do BigQuery

In [8]:
project_id = 'directed-mender-489100-m3'
dataset_id = 'games_data'           # Camada RAW
client = bigquery.Client(project=project_id)

## 3. Download dos Dados (Kaggle)

In [9]:
print("Buscando base de dados no Kaggle...")
data_path = kagglehub.dataset_download("gabrigoncam/video-game-dataset-multidimensional")

all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"\n{len(all_files)} arquivos encontrados.")

Buscando base de dados no Kaggle...
Using Colab cache for faster access to the 'video-game-dataset-multidimensional' dataset.

11 arquivos encontrados.


## 4. Carregamento Inicial (CSV -> DataFrame)

In [10]:
data_frames = {} # Dicionário para armazenar as tabelas

for file_path in all_files:
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')

    print(f"Lendo {file_name}...")

    # Tratamentos específicos de leitura
    if file_name == 'games.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    elif file_name == 'game_platforms.csv':
        df = pd.read_csv(file_path, dtype={'game_id': str})
    elif file_name == 'game_developers.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    else:
        df = pd.read_csv(file_path)

    data_frames[table_name] = df

print("\nCarga concluída em memória.")

Lendo game_tags.csv...
Lendo game_developers.csv...
Lendo game_metacritic_platforms.csv...
Lendo game_genres.csv...
Lendo games.csv...
Lendo game_status.csv...
Lendo game_platforms.csv...
Lendo game_derived_metrics.csv...
Lendo game_ratings.csv...
Lendo game_stores.csv...
Lendo game_publishers.csv...

Carga concluída em memória.


## 5. Limpeza de Dados e Tratamento de Nulos (NaN)

In [11]:
for table_name in list(data_frames.keys()):
    print(f"\n--- Limpando Tabela: {table_name} ---")
    df = data_frames[table_name]
    initial_rows = len(df)

    # 1. Análise de Nulos (Top 5 colunas com NaN)
    null_info = df.isnull().sum()
    null_info = null_info[null_info > 0].sort_values(ascending=False)
    if not null_info.empty:
        print(f"   >> Colunas com valores nulos encontrados:\n{null_info.head(5)}")

    # 2. Remoção de Duplicatas
    df = df.drop_duplicates()
    rows_after_dupes = len(df)

    # 3. Tratamento de Nulos Específicos
    # Remover linhas onde ID ou Nome sejam nulos (campos obrigatórios)
    id_col = 'id' if 'id' in df.columns else (df.columns[0] if 'id' in df.columns[0].lower() else None)
    if id_col:
        df = df.dropna(subset=[id_col])

    # Preencher notas numéricas com 0 (BI)
    numeric_cols_to_fill = ['metacritic', 'rating', 'playtime', 'ratings_count', 'reviews_count']
    for col in numeric_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Preencher strings vazias com placeholder
    string_cols_to_fill = ['description_short', 'website', 'metacritic_url', 'tba']
    for col in string_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna("Sem Informação")

    rows_final = len(df)
    data_frames[table_name] = df

    print(f"   >> Duplicatas removidas: {initial_rows - rows_after_dupes}")
    print(f"   >> Linhas com ID nulo removidas: {rows_after_dupes - rows_final}")
    print(f"   >> Total Final: {rows_final} linhas.")


--- Limpando Tabela: game_tags ---
   >> Colunas com valores nulos encontrados:
tag_language    13
tag_name         5
tag_slug         5
dtype: int64
   >> Duplicatas removidas: 0
   >> Linhas com ID nulo removidas: 0
   >> Total Final: 4370881 linhas.

--- Limpando Tabela: game_developers ---
   >> Colunas com valores nulos encontrados:
developer_name    16
developer_slug     6
dtype: int64
   >> Duplicatas removidas: 0
   >> Linhas com ID nulo removidas: 0
   >> Total Final: 987628 linhas.

--- Limpando Tabela: game_metacritic_platforms ---
   >> Colunas com valores nulos encontrados:
metacritic_url    5
dtype: int64
   >> Duplicatas removidas: 0
   >> Linhas com ID nulo removidas: 0
   >> Total Final: 8118 linhas.

--- Limpando Tabela: game_genres ---
   >> Duplicatas removidas: 0
   >> Linhas com ID nulo removidas: 0
   >> Total Final: 1043171 linhas.

--- Limpando Tabela: games ---
   >> Colunas com valores nulos encontrados:
metacritic_url    815888
metacritic        814978
esrb

## 6. Exploração (EDA)

In [12]:
for table_name, df in data_frames.items():
    print(f"\n--- Prévia: {table_name} ---")
    display(df.head(3))
    df.info(verbose=False, memory_usage='deep')


--- Prévia: game_tags ---


,game_id,tag_id,tag_name,tag_slug,tag_language
0,1000063,31,Singleplayer,singleplayer,eng
1,1000063,40836,Full controller support,full-controller-support,eng
2,1000063,24,RPG,rpg,eng


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4370881 entries, 0 to 4370880
Columns: 5 entries, game_id to tag_language
dtypes: int64(2), object(3)
memory usage: 762.5 MB

--- Prévia: game_developers ---


,game_id,developer_id,developer_name,developer_slug
0,1000063,462470,Muteppo Software,muteppo-software
1,100004,40739,alegaroficial,alegaroficial
2,1000067,258713,Cute Hannah's Games,cute-hannahs-games


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987628 entries, 0 to 987627
Columns: 4 entries, game_id to developer_slug
dtypes: int64(2), object(2)
memory usage: 128.6 MB

--- Prévia: game_metacritic_platforms ---


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119508 entries, 0 to 119507
Columns: 5 entries, game_id to rating_percent
dtypes: float64(1), int64(3), object(1)
memory usage: 10.1 MB

--- Prévia: game_stores ---


,game_id,store_id,store_name,store_slug,store_url
0,1000063,1,Steam,steam,https://store.steampowered.com/app/3544090/
1,100004,9,itch.io,itch,https://alegaroficial.itch.io/hippiesvscops
2,1000067,1,Steam,steam,https://store.steampowered.com/app/3582350/


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 889724 entries, 0 to 889723
Columns: 5 entries, game_id to store_url
dtypes: int64(2), object(3)
memory usage: 187.8 MB

--- Prévia: game_publishers ---


,game_id,publisher_id,publisher_name,publisher_slug
0,1000063,84664,Muteppo Software,muteppo-software
1,1000067,49231,Cute Hannah's Games,cute-hannahs-games
2,1000069,47009,Challenging Games,challenging-games


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234990 entries, 0 to 234989
Columns: 4 entries, game_id to publisher_slug
dtypes: int64(2), object(2)
memory usage: 31.9 MB


## 7. Ingestão para o BigQuery

In [13]:
for table_name, df in data_frames.items():
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    print(f"Enviando {table_name} para {table_id}...")

    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()

    print(f"✅ Sucesso! Tabela {table_name} carregada.\n")

Enviando game_tags para directed-mender-489100-m3.games_data.game_tags...
✅ Sucesso! Tabela game_tags carregada.

Enviando game_developers para directed-mender-489100-m3.games_data.game_developers...
✅ Sucesso! Tabela game_developers carregada.

Enviando game_metacritic_platforms para directed-mender-489100-m3.games_data.game_metacritic_platforms...
✅ Sucesso! Tabela game_metacritic_platforms carregada.

Enviando game_genres para directed-mender-489100-m3.games_data.game_genres...
✅ Sucesso! Tabela game_genres carregada.

Enviando games para directed-mender-489100-m3.games_data.games...
✅ Sucesso! Tabela games carregada.

Enviando game_status para directed-mender-489100-m3.games_data.game_status...
✅ Sucesso! Tabela game_status carregada.

Enviando game_platforms para directed-mender-489100-m3.games_data.game_platforms...
✅ Sucesso! Tabela game_platforms carregada.

Enviando game_derived_metrics para directed-mender-489100-m3.games_data.game_derived_metrics...
✅ Sucesso! Tabela game_de